# Pricing Project

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize
from scipy.special import logsumexp

# Load data
candidate_paths = [Path('data.csv'), Path('Data/data.csv')]
data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Could not find data.csv or Data/data.csv')

df = pd.read_csv(data_path)

# 8 hotel features
feature_cols = [
    'prop_starrating', 'prop_review_score', 'prop_brand_bool',
    'prop_location_score', 'prop_accesibility_score',
    'prop_log_historical_price', 'price_usd', 'promotion_flag'
]
required_cols = ['srch_id', 'booking_bool'] + feature_cols
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise KeyError(f'Missing required columns: {missing_cols}')

estimation_df = df.dropna(subset=required_cols).copy()
dropped_rows = len(df) - len(estimation_df)
if dropped_rows > 0:
    print(f'Dropped {dropped_rows} rows with missing required values.')

invalid_searches = estimation_df.groupby('srch_id')['booking_bool'].sum()
if (invalid_searches > 1).any():
    raise ValueError('Each search must have at most one booked hotel.')

# Normalize features so coefficients are comparable across attributes
means = estimation_df[feature_cols].mean()
stds = estimation_df[feature_cols].std()
constant_cols = stds[stds == 0].index.tolist()
if constant_cols:
    raise ValueError(f'Constant features cannot be estimated: {constant_cols}')

norm_cols = [f'{col}_norm' for col in feature_cols]
estimation_df[norm_cols] = (estimation_df[feature_cols] - means) / stds

# Group by search: each search = one customer seeing a set of hotels
search_data = []
for _, group in estimation_df.groupby('srch_id'):
    X = group[norm_cols].to_numpy(dtype=float)
    y = group['booking_bool'].to_numpy(dtype=float)
    search_data.append((X, y))

def neg_log_likelihood_and_grad(beta):
    b0, b = beta[0], beta[1:]
    nll = 0.0
    grad = np.zeros_like(beta)

    for X, y in search_data:
        u = b0 + X @ b
        log_denom = logsumexp(np.concatenate(([0.0], u)))
        probs = np.exp(u - log_denom)
        diff = probs - y

        nll += log_denom - np.dot(y, u)
        grad[0] += diff.sum()
        grad[1:] += X.T @ diff

    return nll, grad

def neg_log_likelihood(beta):
    return neg_log_likelihood_and_grad(beta)[0]

def neg_log_likelihood_grad(beta):
    return neg_log_likelihood_and_grad(beta)[1]

# Optimize
initial_beta = np.zeros(len(feature_cols) + 1)
result = minimize(
    neg_log_likelihood,
    initial_beta,
    jac=neg_log_likelihood_grad,
    method='L-BFGS-B'
)

if not result.success:
    raise RuntimeError(f'Optimization failed: {result.message}')

beta_star = result.x

# Print results
names = ['intercept'] + feature_cols
print(f'Rows used: {len(estimation_df)}')
print(f'Searches used: {estimation_df["srch_id"].nunique()}')
print(f'Bookings: {int(estimation_df["booking_bool"].sum())}')
print(f'No-purchase searches: {int((invalid_searches == 0).sum())}')
print('\nEstimated MNL coefficients:')
for name, val in zip(names, beta_star):
    print(f'  {name:35s} {val:+.6f}')
print(f'\nLog-likelihood: {-result.fun:.4f}')
print(f'Optimizer status: {result.message}')

pretty_names = {
    'prop_starrating': 'star rating',
    'prop_review_score': 'review score',
    'prop_brand_bool': 'brand indicator',
    'prop_location_score': 'location score',
    'prop_accesibility_score': 'accessibility score',
    'prop_log_historical_price': 'historical price',
    'price_usd': 'displayed price',
    'promotion_flag': 'promotion flag'
}

print('\nCoefficient interpretation (holding other features fixed):')
if beta_star[0] < 0:
    print('  intercept: negative, so the outside option is attractive on average.')
else:
    print('  intercept: positive, so booking a displayed hotel is attractive on average.')

for name, val in zip(feature_cols, beta_star[1:]):
    direction = 'raises' if val > 0 else 'lowers'
    print(
        f'  {pretty_names[name]:35s} {direction} hotel utility by {abs(val):.3f} '
        'for a 1 standard deviation increase.'
    )


Rows used: 153009
Searches used: 8354
Bookings: 5848
No-purchase searches: 2506

Estimated MNL coefficients:
  intercept                           -1.746680
  prop_starrating                     +0.408132
  prop_review_score                   +0.108754
  prop_brand_bool                     +0.101381
  prop_location_score                 +0.021995
  prop_accesibility_score             +0.043445
  prop_log_historical_price           -0.066872
  price_usd                           -1.331110
  promotion_flag                      +0.159765

Log-likelihood: -20611.3260
Optimizer status: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

Coefficient interpretation (holding other features fixed):
  intercept: negative, so the outside option is attractive on average.
  star rating                         raises hotel utility by 0.408 for a 1 standard deviation increase.
  review score                        raises hotel utility by 0.109 for a 1 standard deviation increase.
  brand indicator 

In [3]:
# Problem 2: Assortment Optimization under MNL
# Using the MNL estimated in Problem 1 (beta_star) and the same standardization (means/stds).

from pathlib import Path

small_data_paths = {
    "data1": [Path("Data/data1.csv"), Path("data1.csv")],
    "data2": [Path("Data/data2.csv"), Path("data2.csv")],
    "data3": [Path("Data/data3.csv"), Path("data3.csv")],
    "data4": [Path("Data/data4.csv"), Path("data4.csv")],
}


def load_small_dataset(name: str) -> pd.DataFrame:
    candidates = small_data_paths[name]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        raise FileNotFoundError(f"Could not find {name}.csv in expected locations: {candidates}")
    return pd.read_csv(path)


def compute_mnl_weights(df_small: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """Return (v, price, df_with_norm_cols). v_j = exp(u_j), u_j = b0 + x_norm @ b."""
    missing = [c for c in feature_cols if c not in df_small.columns]
    if missing:
        raise KeyError(f"Missing columns in small dataset: {missing}")

    df2 = df_small.copy()
    # Apply the same standardization as in Problem 1
    X_norm = (df2[feature_cols] - means) / stds

    b0, b = float(beta_star[0]), beta_star[1:].astype(float)
    u = b0 + X_norm.to_numpy(dtype=float) @ b
    v = np.exp(u)
    price = df2["price_usd"].to_numpy(dtype=float)
    return v, price, df2


def expected_revenue(v: np.ndarray, price: np.ndarray, mask: np.ndarray) -> float:
    """Expected revenue for offering subset S = {j: mask_j==True} under MNL with outside option."""
    vS = v[mask]
    pS = price[mask]
    denom = 1.0 + vS.sum()
    return float((pS * vS).sum() / denom)


def optimal_assortment_revenue_ordered(v: np.ndarray, price: np.ndarray) -> tuple[np.ndarray, float]:
    """Find optimal assortment using the MNL 'revenue-ordered assortment' property.

    We evaluate assortments of the form S(r) = {j : price_j >= r} for each distinct price level r.
    This avoids cutting through ties.

    Returns (mask, best_rev).
    """
    n = len(price)
    order = np.argsort(-price, kind="mergesort")  # stable sort for reproducibility
    p_sorted = price[order]
    v_sorted = v[order]

    cum_v = np.cumsum(v_sorted)
    cum_pv = np.cumsum(p_sorted * v_sorted)

    # Candidate cut points: end of each price level (include all items with price >= that level)
    cut_idxs = np.flatnonzero(np.r_[p_sorted[1:] != p_sorted[:-1], True])

    best_rev = 0.0
    best_k = -1  # -1 means empty assortment

    for k in cut_idxs:
        denom = 1.0 + cum_v[k]
        rev = float(cum_pv[k] / denom)
        if rev > best_rev:
            best_rev = rev
            best_k = int(k)

    best_mask_sorted = np.zeros(n, dtype=bool)
    if best_k >= 0:
        best_mask_sorted[: best_k + 1] = True

    best_mask = np.zeros(n, dtype=bool)
    best_mask[order] = best_mask_sorted
    return best_mask, best_rev


results = []
for name in ["data1", "data2", "data3", "data4"]:
    df_small = load_small_dataset(name)
    v, price, _ = compute_mnl_weights(df_small)
    mask, rev = optimal_assortment_revenue_ordered(v, price)

    results.append(
        {
            "dataset": name,
            "n_hotels": int(len(df_small)),
            "n_offered": int(mask.sum()),
            "expected_revenue": rev,
        }
    )

    offered_idx = np.flatnonzero(mask)
    offered_prices = price[mask]

    print(f"\n{name}: optimal offered subset size = {mask.sum()} out of {len(df_small)}")
    print(f"{name}: expected revenue = {rev:.6f}")
    print(f"{name}: offered row indices (0-based) = {offered_idx.tolist()}")
    print(f"{name}: offered prices = {np.round(np.sort(offered_prices), 2).tolist()}")

summary_df = pd.DataFrame(results).sort_values("dataset")
print("\nSummary")
display(summary_df)


data1: optimal offered subset size = 18 out of 27
data1: expected revenue = 107.352012
data1: offered row indices (0-based) = [0, 1, 2, 3, 4, 5, 6, 12, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26]
data1: offered prices = [108.0, 112.0, 118.0, 120.0, 121.0, 121.0, 125.0, 131.0, 131.0, 140.0, 144.0, 145.0, 147.0, 147.0, 150.0, 154.0, 156.0, 158.0]

data2: optimal offered subset size = 10 out of 29
data2: expected revenue = 131.110898
data2: offered row indices (0-based) = [0, 1, 6, 7, 8, 9, 10, 21, 23, 25]
data2: offered prices = [133.0, 134.0, 137.0, 143.0, 146.0, 147.0, 161.0, 206.0, 212.0, 233.0]

data3: optimal offered subset size = 18 out of 26
data3: expected revenue = 121.053221
data3: offered row indices (0-based) = [0, 1, 2, 3, 4, 5, 7, 8, 10, 11, 13, 14, 15, 16, 18, 19, 23, 24]
data3: offered prices = [123.0, 128.0, 129.0, 138.0, 138.0, 140.0, 144.0, 150.0, 153.0, 171.0, 180.0, 181.0, 190.0, 191.0, 195.0, 211.0, 281.0, 603.0]

data4: optimal offered subset size = 11 out of 27
data4

,dataset,n_hotels,n_offered,expected_revenue
0,data1,27,18,107.352012
1,data2,29,10,131.110898
2,data3,26,18,121.053221
3,data4,27,11,97.407947


In [ ]:
# Problem 3: Pricing under MNL
# We display ALL hotels and choose prices to maximize expected revenue.

price_col = "price_usd"
price_idx = feature_cols.index(price_col)

beta0 = float(beta_star[0])
beta = beta_star[1:].astype(float)
beta_price = float(beta[price_idx])
mean_price = float(means[price_col])
std_price = float(stds[price_col])


def optimize_prices_mnl(df_small: pd.DataFrame, *, lower: float, upper: float) -> dict:
    missing = [c for c in feature_cols if c not in df_small.columns]
    if missing:
        raise KeyError(f"Missing columns in small dataset: {missing}")

    X = df_small[feature_cols].to_numpy(dtype=float)

    # Standardize non-price features using Problem 1 means/stds
    z = (X - means[feature_cols].to_numpy(dtype=float)) / stds[feature_cols].to_numpy(dtype=float)

    # Split out price term so we can vary p
    z_no_price = z.copy()
    z_no_price[:, price_idx] = 0.0

    # u(p) = base + beta_price * ((p - mean_price)/std_price)
    base = beta0 + z_no_price @ beta

    def neg_rev_and_grad(p: np.ndarray) -> tuple[float, np.ndarray]:
        # enforce numeric dtype
        p = p.astype(float)
        u = base + beta_price * ((p - mean_price) / std_price)
        v = np.exp(u)
        D = 1.0 + v.sum()
        N = float((p * v).sum())
        rev = N / D

        # gradient
        c = beta_price / std_price  # dv_j/dp_j = c * v_j
        dv = c * v
        dN = v + p * dv
        dD = dv
        grad = (dN * D - N * dD) / (D * D)

        return -rev, -grad

    def neg_rev(p: np.ndarray) -> float:
        return neg_rev_and_grad(p)[0]

    def neg_rev_grad(p: np.ndarray) -> np.ndarray:
        return neg_rev_and_grad(p)[1]

    p0 = df_small[price_col].to_numpy(dtype=float)
    bounds = [(lower, upper) for _ in range(len(p0))]

    res = minimize(
        neg_rev,
        p0,
        jac=neg_rev_grad,
        bounds=bounds,
        method="L-BFGS-B",
    )

    if not res.success:
        raise RuntimeError(f"Pricing optimization failed: {res.message}")

    p_star = res.x.astype(float)

    # baseline revenue at current prices
    base_u0 = base + beta_price * ((p0 - mean_price) / std_price)
    v0 = np.exp(base_u0)
    rev0 = float((p0 * v0).sum() / (1.0 + v0.sum()))

    # optimized revenue
    base_u_star = base + beta_price * ((p_star - mean_price) / std_price)
    v_star = np.exp(base_u_star)
    rev_star = float((p_star * v_star).sum() / (1.0 + v_star.sum()))

    return {
        "p0": p0,
        "p_star": p_star,
        "rev0": rev0,
        "rev_star": rev_star,
        "optimizer_message": res.message,
        "n_iters": int(res.nit),
    }


pricing_results = []
for name in ["data1", "data2", "data3", "data4"]:
    df_small = load_small_dataset(name)

    p0 = df_small[price_col].to_numpy(dtype=float)
    lo = 1.0
    hi = max(1000.0, 3.0 * float(np.max(p0)))

    out = optimize_prices_mnl(df_small, lower=lo, upper=hi)

    pricing_results.append(
        {
            "dataset": name,
            "n_hotels": int(len(df_small)),
            "rev_current": out["rev0"],
            "rev_opt": out["rev_star"],
            "rev_lift": out["rev_star"] - out["rev0"],
            "price_min_current": float(np.min(out["p0"])),
            "price_max_current": float(np.max(out["p0"])),
            "price_min_opt": float(np.min(out["p_star"])),
            "price_max_opt": float(np.max(out["p_star"])),
            "iters": out["n_iters"],
        }
    )

    print(f"\n{name}: expected revenue (current prices) = {out['rev0']:.6f}")
    print(f"{name}: expected revenue (optimal prices) = {out['rev_star']:.6f}")
    print(f"{name}: optimizer status = {out['optimizer_message']}")
    print(f"{name}: optimal prices (rounded) = {np.round(out['p_star'], 2).tolist()}")

pricing_summary = pd.DataFrame(pricing_results).sort_values("dataset")
print("\nPricing summary")
display(pricing_summary)